# 00. Inventory and data dictionary

Цель: автоматически проинвентаризировать `../data`, оценить временное покрытие и явно зафиксировать отсутствующие источники.

Принципы: исходные файлы не изменяются; неизвестные теги не расшифровываются по догадке; `T/P/F/L` используются только как слабая категория по префиксу, а не как технологическая расшифровка.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown

HERE = Path.cwd().resolve()
EDA_DIR = HERE if HERE.name == 'eda' else HERE / 'eda'
DATA_DIR = EDA_DIR.parent / 'data'
ARTIFACTS = EDA_DIR / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(EDA_DIR))
from eda_utils import source_inventory, load_telemetry, timestamp_audit, PREFIX_CATEGORY
assert DATA_DIR.is_dir(), f'Data directory not found: {DATA_DIR}'
pd.set_option('display.max_columns', 100)
print('DATA_DIR:', DATA_DIR)

## Автоматический inventory

In [ ]:
inventory, expected = source_inventory(DATA_DIR)
display(inventory)
display(expected.style.map(lambda v: 'background:#ffd6d6' if v is False else '', subset=['available_in_data']))
inventory.to_csv(ARTIFACTS / 'source_inventory.csv', index=False)
expected.to_csv(ARTIFACTS / 'expected_source_availability.csv', index=False)
px.bar(inventory, x='source', y='size_mb', color='format', title='Размер источников в data/ (MB)').show()

## Покрытие временной оси

Интервалы оцениваются по timestamp. Никакого объединения по номеру строки.

In [ ]:
audits = []
for path in sorted(DATA_DIR.glob('*.csv')):
    df = load_telemetry(path)
    audits.append(timestamp_audit(df, path.stem))
time_inventory = pd.DataFrame(audits)
display(time_inventory)
time_inventory.to_csv(ARTIFACTS / 'timestamp_inventory.csv', index=False)
timeline = time_inventory.melt(id_vars='source', value_vars=['start', 'end'], var_name='boundary', value_name='timestamp')
px.timeline(time_inventory, x_start='start', x_end='end', y='source', color='source', title='Временное покрытие').show()

## Честный data dictionary

Без `Теги_хакатон*.xlsx` в `data/` поля description/unit/installation остаются `UNKNOWN`. Префиксная категория — только техническая гипотеза для навигации.

In [ ]:
rows = []
for path in sorted(DATA_DIR.glob('*.csv')):
    cols = [c for c in load_telemetry(path).columns if c != 'date']
    installation = 'АВТ' if path.name == 'avt_tags.csv' else '24-2000 / гидроочистка' if path.name == '242000_tags.csv' else 'UNKNOWN'
    for tag in cols:
        prefix = ''.join(ch for ch in tag if ch.isalpha()).upper()
        rows.append({'tag': tag, 'installation': installation, 'description': 'UNKNOWN',
                     'measurement_type': PREFIX_CATEGORY.get(prefix, 'unknown'), 'unit': 'UNKNOWN',
                     'source': path.name, 'category': PREFIX_CATEGORY.get(prefix, 'unknown')})
dictionary = pd.DataFrame(rows)
display(dictionary)
dictionary.to_csv(ARTIFACTS / 'data_dictionary_without_reference.csv', index=False)
px.histogram(dictionary, x='category', color='installation', barmode='group', title='Теги по префиксной категории').show()

## Gate для следующих этапов

Если ЛИМС/ПАК/справочник отсутствуют, то нельзя корректно выполнить: LIMS–PAK comparison, AGE/FRESHNESS, анализ серы относительно 10 мг/кг, target suitability, воспроизведение ВАК и физически осмысленный feature→target lag analysis. Ниже — машинно-читаемый статус.

In [ ]:
gates = pd.DataFrame([
    {'analysis': 'LIMS structure / targets / sulfur', 'ready': bool(expected.query("expected_source == 'lims'")['available_in_data'].iloc[0]), 'reason': 'Requires LIMS in data/'},
    {'analysis': 'PAK diagnostics and LIMS–PAK', 'ready': bool(expected.query("expected_source == 'pak'")['available_in_data'].iloc[0]), 'reason': 'Requires PAK in data/'},
    {'analysis': 'Physical tag semantics / controls / VAK', 'ready': bool(expected.query("expected_source == 'tag_dictionary'")['available_in_data'].iloc[0]), 'reason': 'Requires tag dictionary in data/'},
])
display(gates)
gates.to_csv(ARTIFACTS / 'analysis_gates.csv', index=False)